Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

In [ ]:
it = iter(dataset)
next(it)

{'text': "How AP reported in all formats from tornado-stricken regionsMarch 8, 2012\nWhen the first serious bout of tornadoes of 2012 blew through middle America in the middle of the night, they touched down in places hours from any AP bureau. Our closest video journalist was Chicago-based Robert Ray, who dropped his plans to travel to Georgia for Super Tuesday, booked several flights to the cities closest to the strikes and headed for the airport. He’d decide once there which flight to take.\nHe never got on board a plane. Instead, he ended up driving toward Harrisburg, Ill., where initial reports suggested a town was destroyed. That decision turned out to be a lucky break for the AP. Twice.\nRay was among the first journalists to arrive and he confirmed those reports -- in all formats. He shot powerful video, put victims on the phone with AP Radio and played back sound to an editor who transcribed the interviews and put the material on text wires. He then walked around the devastatio

ingest.py

```
# configs/ingest.yaml
shard_size_docs: 10000
target_tokens: 200000000
output_dir: data/raw
```

In [ ]:



output_path

PosixPath('data/raw/shard_00008.parquet')

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Ingesting: 50.3ktok [00:08, 6.16ktok/s, docs=80, shard=00007]                         

[Shard 00000] [Shard 00000] docs=10 | tokens=4,203
[Shard 00001] [Shard 00001] docs=20 | tokens=7,684
[Shard 00002] [Shard 00002] docs=30 | tokens=9,808
[Shard 00003] [Shard 00003] docs=40 | tokens=14,981
[Shard 00004] [Shard 00004] docs=50 | tokens=19,797
[Shard 00005] [Shard 00005] docs=60 | tokens=31,857
[Shard 00006] [Shard 00006] docs=70 | tokens=41,374
[Shard 00007] [Shard 00007] docs=80 | tokens=45,671
[Shard 00008] [Shard 00008] docs=89 | tokens=50,259


list

In [ ]:
from datetime import datetime



'2026-08-01T12:28:04.672909Z'

In [17]:
list(dataset.take(4))

[{'text': "How AP reported in all formats from tornado-stricken regionsMarch 8, 2012\nWhen the first serious bout of tornadoes of 2012 blew through middle America in the middle of the night, they touched down in places hours from any AP bureau. Our closest video journalist was Chicago-based Robert Ray, who dropped his plans to travel to Georgia for Super Tuesday, booked several flights to the cities closest to the strikes and headed for the airport. He’d decide once there which flight to take.\nHe never got on board a plane. Instead, he ended up driving toward Harrisburg, Ill., where initial reports suggested a town was destroyed. That decision turned out to be a lucky break for the AP. Twice.\nRay was among the first journalists to arrive and he confirmed those reports -- in all formats. He shot powerful video, put victims on the phone with AP Radio and played back sound to an editor who transcribed the interviews and put the material on text wires. He then walked around the devastati

In [23]:
import numpy as np
import pandas as pd
import pyarrow as pa
df = pd.DataFrame({'one': [-1, np.nan, 2.5],
                   'two': ['foo', 'bar', 'baz'],
                   'three': [True, False, True]},
                   index=list('abc'))
table = pa.Table.from_pandas(df)
table

pyarrow.Table
one: double
two: string
three: bool
__index_level_0__: string
----
one: [[-1,null,2.5]]
two: [["foo","bar","baz"]]
three: [[true,false,true]]
__index_level_0__: [["a","b","c"]]

In [24]:
import pyarrow.parquet as pq
pq.write_table(table, 'test.parquet')

In [27]:
table2 = pq.read_table('test.parquet')
table2.to_pandas()

,one,two,three
a,-1.0,foo,True
b,NaN,bar,False
c,2.5,baz,True


The ingestion loop is essentially:

In [ ]:
buffer = []

for sample in fineweb:
    buffer.append(sample)

    if len(buffer) == 10_000:
        table = pa.Table.from_pylist(buffer, schema=schema)
        pq.write_table(table, output_path)
        buffer.clear()

Everything else in ingest.py—counting tokens, naming shards, stopping at your target budget, logging progress—is pipeline logic. PyArrow's role is just to convert your buffered Python dictionaries into an efficient Parquet file. That's why the guidance says to focus only on schema, table creation, and write_table.